## read data

In [1]:
import torch
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# 1. 你的实际文件路径
file_path = r"\\cabinet\derivatives\cheXpert\FVs_medsiglip"

print("正在使用 PyTorch 加载数据...")
data_dict = torch.load(file_path, map_location='cpu', weights_only=False)

正在使用 PyTorch 加载数据...


## Check data structure

In [17]:
import json
import numpy as np

def summarize_obj(obj, max_preview=5):
    """
    递归地把复杂对象整理成可写入 JSON 的结构摘要
    """
    # dict
    if isinstance(obj, dict):
        return {
            "type": "dict",
            "num_keys": len(obj),
            "keys": list(obj.keys()),
            "items": {k: summarize_obj(v, max_preview) for k, v in obj.items()}
        }

    # numpy array
    elif isinstance(obj, np.ndarray):
        summary = {
            "type": "numpy.ndarray",
            "shape": list(obj.shape),
            "dtype": str(obj.dtype)
        }

        # # 给一点预览，避免整个数组写进去
        # try:
        #     if obj.ndim == 1:
        #         summary["preview"] = obj[:max_preview].tolist()
        #     elif obj.ndim >= 2:
        #         summary["preview"] = obj[:max_preview].tolist()
        # except:
        #     summary["preview"] = "preview_failed"

        return summary

    # list / tuple
    elif isinstance(obj, (list, tuple)):
        return {
            "type": type(obj).__name__,
            "length": len(obj),
            # "preview": [summarize_obj(x, max_preview) for x in obj[:max_preview]]
        }

    # numpy scalar
    elif isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()

    # 普通 Python 基本类型
    elif isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj

    # 其他类型
    else:
        return {
            "type": str(type(obj)),
            "repr": repr(obj)
        }


# 假设你的数据叫 data_dict
summary = summarize_obj(data_dict)

# 保存成 JSON 文件
with open("data_structure_summary_no_pre.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("已保存为 data_structure_summary_no_pre.json")

已保存为 data_structure_summary_no_pre.json


## check findings structure

In [2]:
import numpy as np

findings = data_dict["train_val"]["remarks"]["findings"]

print("type:", type(findings))
print("shape:", findings.shape)
print("dtype:", findings.dtype)

print("\n前20个样本：")
print(findings[:20])

unique_findings = np.unique(findings)
print("\n唯一值数量:", len(unique_findings))
print("前50个唯一值：")
print(unique_findings[:50])

for sep in ['|', ';', ',']:
    count = np.sum([sep in x for x in findings])
    print(f"\n包含分隔符 '{sep}' 的样本数:", count)

type: <class 'numpy.ndarray'>
shape: (86524,)
dtype: <U100

前20个样本：
['Cardiomegaly' 'Cardiomegaly|Emphysema' 'Cardiomegaly|Effusion'
 'No Finding' 'Mass|Nodule' 'No Finding' 'No Finding' 'No Finding'
 'No Finding' 'No Finding' 'No Finding' 'Infiltration'
 'Effusion|Infiltration' 'No Finding' 'No Finding' 'Cardiomegaly'
 'No Finding' 'Nodule' 'Emphysema' 'Infiltration']

唯一值数量: 619
前50个唯一值：
['Atelectasis' 'Atelectasis|Cardiomegaly'
 'Atelectasis|Cardiomegaly|Consolidation'
 'Atelectasis|Cardiomegaly|Consolidation|Edema|Effusion|Infiltration'
 'Atelectasis|Cardiomegaly|Consolidation|Edema|Effusion|Pleural_Thickening'
 'Atelectasis|Cardiomegaly|Consolidation|Effusion'
 'Atelectasis|Cardiomegaly|Consolidation|Effusion|Infiltration'
 'Atelectasis|Cardiomegaly|Consolidation|Effusion|Infiltration|Mass'
 'Atelectasis|Cardiomegaly|Consolidation|Effusion|Mass'
 'Atelectasis|Cardiomegaly|Consolidation|Effusion|Mass|Nodule'
 'Atelectasis|Cardiomegaly|Consolidation|Effusion|Mass|Pleural_Thickening'

## counting label number

In [3]:
import numpy as np
import pandas as pd
from collections import Counter

findings = data_dict["train_val"]["remarks"]["findings"]

# 1) 收集基础标签
counter = Counter()
for f in findings:
    for lab in f.split('|'):
        lab = lab.strip()
        if lab:
            counter[lab] += 1

all_labels = sorted(counter.keys())
print("基础标签数:", len(all_labels))
print(all_labels)
print(counter)

# 2) 转成 multi-hot
label_to_idx = {lab: i for i, lab in enumerate(all_labels)}
Y = np.zeros((len(findings), len(all_labels)), dtype=np.int8)

for i, s in enumerate(findings):
    for lab in s.split('|'):
        lab = lab.strip()
        if lab:
            Y[i, label_to_idx[lab]] = 1

label_df = pd.DataFrame(Y, columns=all_labels)
print(label_df.head())
print(label_df.sum().sort_values(ascending=False))

基础标签数: 15
['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']
Counter({'No Finding': 50500, 'Infiltration': 13782, 'Effusion': 8659, 'Atelectasis': 8280, 'Nodule': 4708, 'Mass': 4034, 'Consolidation': 2852, 'Pneumothorax': 2637, 'Pleural_Thickening': 2242, 'Cardiomegaly': 1707, 'Emphysema': 1423, 'Edema': 1378, 'Fibrosis': 1251, 'Pneumonia': 876, 'Hernia': 141})
   Atelectasis  Cardiomegaly  Consolidation  Edema  Effusion  Emphysema  \
0            0             1              0      0         0          0   
1            0             1              0      0         0          1   
2            0             1              0      0         1          0   
3            0             0              0      0         0          0   
4            0             0              0      0         0          0   

   Fibrosis  Hernia  Infiltratio

In [4]:
ages = data_dict["train_val"]["remarks"]["ages"]
genders = data_dict["train_val"]["remarks"]["genders"]
follow_ups = data_dict["train_val"]["remarks"]["follow_ups"]

print("ages:")
print("min =", ages.min(), "max =", ages.max(), "mean =", ages.mean(), "std =", ages.std())

print("\ngenders:")
print(np.unique(genders, return_counts=True))

print("\nfollow_ups:")
print("min =", follow_ups.min(), "max =", follow_ups.max(), "mean =", follow_ups.mean(), "std =", follow_ups.std())
print("前20个唯一值及频数:")
vals, counts = np.unique(follow_ups, return_counts=True)
print(list(zip(vals[:20], counts[:20])))

ages:
min = 0 max = 95 mean = 46.59862003605936 std = 16.670140842543418

genders:
(array(['F', 'M'], dtype='<U1'), array([38066, 48458]))

follow_ups:
min = 0 max = 108 mean = 5.096620590818732 std = 8.656890502755571
前20个唯一值及频数:
[(np.int64(0), np.int64(28008)), (np.int64(1), np.int64(11878)), (np.int64(2), np.int64(8020)), (np.int64(3), np.int64(6024)), (np.int64(4), np.int64(4781)), (np.int64(5), np.int64(3922)), (np.int64(6), np.int64(3197)), (np.int64(7), np.int64(2640)), (np.int64(8), np.int64(2195)), (np.int64(9), np.int64(1853)), (np.int64(10), np.int64(1569)), (np.int64(11), np.int64(1320)), (np.int64(12), np.int64(1132)), (np.int64(13), np.int64(1000)), (np.int64(14), np.int64(851)), (np.int64(15), np.int64(758)), (np.int64(16), np.int64(671)), (np.int64(17), np.int64(600)), (np.int64(18), np.int64(540)), (np.int64(19), np.int64(480))]


we find that in ages ,min = 0
check 0

In [5]:
ages = data_dict["train_val"]["remarks"]["ages"]

print("age == 0 的样本数:", np.sum(ages == 0))
print("age <= 5 的样本数:", np.sum(ages <= 5))
print("age 的前20个唯一值及频数:")
vals, counts = np.unique(ages, return_counts=True)
print(list(zip(vals[:20], counts[:20])))

age == 0 的样本数: 13
age <= 5 的样本数: 462
age 的前20个唯一值及频数:
[(np.int64(0), np.int64(13)), (np.int64(1), np.int64(53)), (np.int64(2), np.int64(64)), (np.int64(3), np.int64(84)), (np.int64(4), np.int64(104)), (np.int64(5), np.int64(144)), (np.int64(6), np.int64(171)), (np.int64(7), np.int64(205)), (np.int64(8), np.int64(216)), (np.int64(9), np.int64(251)), (np.int64(10), np.int64(296)), (np.int64(11), np.int64(275)), (np.int64(12), np.int64(297)), (np.int64(13), np.int64(420)), (np.int64(14), np.int64(304)), (np.int64(15), np.int64(391)), (np.int64(16), np.int64(543)), (np.int64(17), np.int64(501)), (np.int64(18), np.int64(616)), (np.int64(19), np.int64(645))]


maybe 0 means baby?
let's do PCA first

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

X = data_dict["train_val"]["feature_vectors"]

scaler = StandardScaler()
X_std = scaler.fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_std)

print("前10个主成分方差贡献率:")
print(pca.explained_variance_ratio_[:10])

print("\n前10个累计贡献率:")
print(np.cumsum(pca.explained_variance_ratio_[:10]))

前10个主成分方差贡献率:
[0.167293   0.07061186 0.0508058  0.04991007 0.0450289  0.04294764
 0.03795433 0.03158024 0.02791312 0.02209106]

前10个累计贡献率:
[0.167293   0.23790485 0.28871065 0.33862072 0.38364962 0.42659727
 0.4645516  0.49613184 0.524045   0.5461361 ]


In [7]:
ages = data_dict["train_val"]["remarks"]["ages"].astype(float)
ages[ages == 0] = np.nan
follow_ups = data_dict["train_val"]["remarks"]["follow_ups"]
genders = data_dict["train_val"]["remarks"]["genders"]

for k in range(10):
    pc = X_pca[:, k]

    mask = ~np.isnan(ages)
    corr_age = np.corrcoef(pc[mask], ages[mask])[0, 1]
    corr_fu = np.corrcoef(pc, follow_ups)[0, 1]

    mean_f = pc[genders == 'F'].mean()
    mean_m = pc[genders == 'M'].mean()
    diff_gender = mean_m - mean_f

    print(f"PC{k+1}: corr(age)={corr_age:.4f}, corr(follow_up)={corr_fu:.4f}, mean(F)={mean_f:.4f}, mean(M)={mean_m:.4f}, diff(M-F)={diff_gender:.4f}")

PC1: corr(age)=0.0189, corr(follow_up)=0.4109, mean(F)=0.1080, mean(M)=-0.0848, diff(M-F)=-0.1928
PC2: corr(age)=0.2056, corr(follow_up)=-0.0350, mean(F)=-0.1634, mean(M)=0.1283, diff(M-F)=0.2917
PC3: corr(age)=-0.0078, corr(follow_up)=0.0794, mean(F)=-1.6382, mean(M)=1.2869, diff(M-F)=2.9251
PC4: corr(age)=0.1438, corr(follow_up)=0.0516, mean(F)=-5.3167, mean(M)=4.1765, diff(M-F)=9.4933
PC5: corr(age)=0.0196, corr(follow_up)=0.0474, mean(F)=1.3587, mean(M)=-1.0673, diff(M-F)=-2.4259
PC6: corr(age)=0.2064, corr(follow_up)=-0.1182, mean(F)=-2.9141, mean(M)=2.2892, diff(M-F)=5.2032
PC7: corr(age)=-0.2057, corr(follow_up)=0.0340, mean(F)=-3.2963, mean(M)=2.5894, diff(M-F)=5.8858
PC8: corr(age)=-0.4389, corr(follow_up)=0.0406, mean(F)=-0.0872, mean(M)=0.0685, diff(M-F)=0.1558
PC9: corr(age)=0.1997, corr(follow_up)=0.0650, mean(F)=0.6028, mean(M)=-0.4736, diff(M-F)=-1.0764
PC10: corr(age)=-0.2743, corr(follow_up)=-0.0526, mean(F)=-0.0127, mean(M)=0.0099, diff(M-F)=0.0226
